In [3]:
import sys
sys.path.append('C:/Users/DELL/Desktop/LLM-GNN')
import torch
import config as args
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors
import numpy as np
import Function as Fnc
import networkx as nx
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
model_GCN  = torch.load(
    args.PATH_MODULE_SAVE_GCN,
    weights_only=False
)

In [ ]:
model_GAT  = torch.load(
    args.PATH_MODULE_SAVE_GAT,
    weights_only=False
)

In [ ]:
model_VGAE  = torch.load(
    args.PATH_MODULE_SAVE_VGAE,
    weights_only=False
)

c:\Users\DELL\anaconda3\envs\LLM-GNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
type(model_GCN)

Architectures.GCN.GCN

In [4]:
type(model_GAT)

Architectures.GAT.GAT

In [5]:
type(model_VGAE)

torch_geometric.nn.models.autoencoder.VGAE

In [6]:
model_GCN.eval()

GCN(
  (conv1): GCNConv(3, 16)
  (conv2): GCNConv(16, 32)
  (conv3): GCNConv(32, 16)
  (conv4): GCNConv(16, 8)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [7]:
model_GAT.eval()

GAT(
  (conv1): GATConv(3, 16, heads=8)
  (conv2): GATConv(128, 32, heads=8)
  (conv3): GATConv(256, 16, heads=8)
  (conv4): GATConv(128, 8, heads=1)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [8]:
model_VGAE.eval()

VGAE(
  (encoder): Encoder(
    (conv1): GCNConv(3, 16)
    (conv2): GCNConv(16, 16)
    (conv3): GCNConv(16, 16)
    (conv_mu): GCNConv(16, 8)
    (conv_logstd): GCNConv(16, 8)
  )
  (decoder): InnerProductDecoder()
)

In [19]:
torch.cuda.empty_cache()
data = torch.load(args.PATH_DATA_SAVE, weights_only=False)

In [10]:
with torch.no_grad():
    node_embeddings_GCN = model_GCN.encode(data.x, data.edge_index)

print(node_embeddings_GCN.shape)
torch.save(node_embeddings_GCN, args.PATH_EMBEDDINGS_SAVE_GCN)


torch.Size([3351, 8])


In [12]:
with torch.no_grad():
    node_embeddings_GAT = model_GAT.encode(data.x, data.edge_index)

print(node_embeddings_GAT.shape)
torch.save(node_embeddings_GAT, args.PATH_EMBEDDINGS_SAVE_GAT)

torch.Size([3351, 8])


In [13]:
with torch.no_grad():
    node_embeddings_VGAE, _ = model_VGAE(data.x, data.edge_index)

print(node_embeddings_VGAE.shape)
torch.save(node_embeddings_VGAE, args.PATH_EMBEDDINGS_SAVE_VGAE)

torch.Size([3351, 8])


In [4]:
node_embeddings = torch.load(args.PATH_EMBEDDINGS_SAVE_GCN)

In [5]:
print(node_embeddings.shape)

torch.Size([3351, 8])


3351 nœuds → 3351 villes 🌍

8 dimensions → embedding latent appris par le GNN

Chaque ville est représentée par un vecteur de dimension 8

1️⃣ Normalisation (CRITIQUE)

In [6]:
node_embeddings = F.normalize(node_embeddings, p=2, dim=1)

2️⃣ Création de l’index FAISS

In [93]:
emb_np = node_embeddings.cpu().numpy()

nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine",
    algorithm="auto"
)
nn.fit(emb_np)

distances, indices = nn.kneighbors(emb_np)


In [94]:
distances.shape

(3351, 5)

In [95]:
indices.shape

(3351, 5)

In [96]:
G = nx.read_graphml(args.PATH_DATA)

In [97]:
max_cc = max(nx.connected_components(G), key=len)
G_large = G.subgraph(max_cc).copy()

In [98]:
df = pd.DataFrame([{'node': n, **G_large.nodes[n]} for n in G_large.nodes()])

In [99]:
df

,node,lon,lat,population,country,city_name
0,0,-145.509722,-17.353889,10000,FRENCH_POLYNESIA,Anaa
1,1,-140.950000,-18.066667,10000,FRENCH_POLYNESIA,Hao Island
2,2,-149.600000,-17.550000,26357,FRENCH_POLYNESIA,Papeete
3,3,-135.000000,-23.033333,10000,FRENCH_POLYNESIA,Gambier Island
4,4,-143.657250,-16.584889,10000,FRENCH_POLYNESIA,Makemo
...,...,...,...,...,...,...
3346,3368,147.950000,-5.880000,10000,PAPUA_NEW_GUINEA,Satwag
3347,3301,148.470000,-9.300000,10000,PAPUA_NEW_GUINEA,Gewoya
3348,3370,146.820000,-6.100000,10000,PAPUA_NEW_GUINEA,Derim
3349,3371,146.600000,-6.133333,10000,PAPUA_NEW_GUINEA,Yalumet


In [100]:
cities = []

for _, row in df.iterrows():
    cities.append({
        "name": row.get("city_name"),
        "country": row.get("country"),
        "population": float(row.get("population", 0)),
        "lat": float(row.get("lat")),
        "lon": float(row.get("lon"))
    })


In [101]:
cities

[{'name': 'Anaa',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -17.35388888888889,
  'lon': -145.50972222222222},
 {'name': 'Hao Island',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -18.066666666666666,
  'lon': -140.95},
 {'name': 'Papeete',
  'country': 'FRENCH_POLYNESIA',
  'population': 26357.0,
  'lat': -17.55,
  'lon': -149.6},
 {'name': 'Gambier Island',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -23.033333333333335,
  'lon': -135.0},
 {'name': 'Makemo',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -16.584888888888887,
  'lon': -143.65725},
 {'name': 'Auckland',
  'country': 'NEW_ZEALAND',
  'population': 417910.0,
  'lat': -37.00805555555556,
  'lon': 174.79166666666666},
 {'name': 'Atuona',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -9.8,
  'lon': -139.03333333333333},
 {'name': 'Bora Bora',
  'country': 'FRENCH_POLYNESIA',
  'population': 10000.0,
  'lat': -16.45

In [102]:
len(cities)

3351

In [103]:
assert len(cities) == node_embeddings.shape[0]


In [104]:
def retrieve_similar_cities(city_index, cities, indices, k=20):
    neighbors = indices[city_index][1 : min(k+1, len(cities))]
    results = []

    for idx in neighbors:
        city = cities[idx]
        text = (
            f"Ville : {city['name']}. "
            f"Pays : {city['country']}. "
            f"Population : {city['population']}. "
            f"Latitude : {city['lat']}, Longitude : {city['lon']}."
        )
        results.append(text)

    return results

In [105]:
city_idx = 150

contexts = Fnc.retrieve_similar_cities(city_idx, cities, indices)

for c in contexts:
    print(c)


Ville : Calgary. Pays : CANADA. Population : 1019942.0. Latitude : 51.11388888888889, Longitude : -114.02.
Ville : San Salvador. Pays : EL_SALVADOR. Population : 525990.0. Latitude : 13.7, Longitude : -89.11666666666666.
Ville : Iquique. Pays : CHILE. Population : 191468.0. Latitude : -20.533333333333335, Longitude : -70.18333333333334.
Ville : Mucuri. Pays : BRAZIL. Population : 26775.0. Latitude : -18.049166666666668, Longitude : -39.86527777777778.


In [106]:
len(contexts)

4

In [107]:
context_text = "\n".join(contexts)

In [ ]:
prompt = f"""
Tu es un expert en analyse de graphes urbains et en similarité entre villes.

Ville de référence :
San Jose (USA), Population : 35768, Latitude : 37.36186194444445, Longitude : -121.9290088888889

Villes candidates :
{context_text}

Tâche :
1. Compare chaque ville candidate à San Jose en utilisant **3 critères** :
   - Pays : si le pays est le même, la similarité est plus élevée
   - Population : si la population est proche (moins de 10x d’écart), la similarité est plus élevée
   - Distance géographique : plus la distance est courte, plus la similarité est élevée
2. Classe les villes de la PLUS similaire à la MOINS similaire
3. Fournis un raisonnement clair pour chaque ville en indiquant **population, distance et pays**

Réponse OBLIGATOIRE dans ce format EXACT :

Ville la plus similaire :
- Nom :
- Justification : (expliquer pourquoi selon population, distance et pays)

Ville intermédiaire :
- Nom :
- Justification : (expliquer pourquoi selon population, distance et pays)

Ville la moins similaire :
- Nom :
- Justification : (expliquer pourquoi selon population, distance et pays)

⚠️ Important : Ne répète jamais le texte des données brutes. Utilise uniquement les informations pertinentes pour justifier la similarité.
Réponds en français clair et structuré.
"""


In [112]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)


response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)




Ville la plus similaire :
- Nom : Mucuri
- Justification : La population de Mucuri (26 775) est très proche de celle de San Jose (35 768), ce qui rend cette ville comparable numériquement. Bien que le pays soit différent (Brésil vs USA), la distance géographique n'est pas extrême (~4 900 km), ce qui en fait la ville la plus similaire parmi les candidates.

Ville intermédiaire :
- Nom : Iquique
- Justification : La population d'Iquique (191 468) est plus grande que celle de San Jose mais reste dans un ordre de grandeur acceptable (<10x). Le pays est différent (Chili), et la distance géographique est assez grande (~9 000 km), ce qui diminue légèrement la similarité.

Ville intermédiaire :
- Nom : San Salvador
- Justification : Population moyenne (525 990), pays différent (El Salvador), mais géographiquement plus proche (~5 500 km). La combinaison de ces facteurs lui donne un score de similarité intermédiaire.

Ville la moins similaire :
- Nom : Calgary
- Justification : La population de